### Import Libraries

In [27]:
# Import necessary libraries and modules
import os
import glob
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from IPython.display import display, clear_output, Markdown
import ipywidgets as widgets

In [9]:
# Load api key from .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

print("API key activated")

API key activated


#### Documents collections

In [10]:
# Load PDF documents from a specified folder
documents = []

for pdf_path in glob.glob("documents/*.pdf"):
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    documents.extend(docs)

print(f"Loaded {len(documents)} PDF Documents.")

Loaded 6 PDF Documents.


#### Text Splitters

In [11]:
# Create splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

# Split documents
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {chunk.page_content}")

Split 6 documents into 22 chunks

Chunk 1: Projects: 
Project 1: Car Price Prediction System 
• Developed a supervised machine learning model to predict car prices using 
historical sales data. 
• Performed data cleaning, feature engineering, model training, and evaluation 
to ensure high predictive accuracy. 
• Deployed the model via Flask for real-time price estimation. 
Technologies: Python, Pandas, Scikit-learn, Flask 
Project 2: Retail Sales Forecasting

Chunk 2: Technologies: Python, Pandas, Scikit-learn, Flask 
Project 2: Retail Sales Forecasting 
• Built a time-series forecasting system to predict monthly sales trends for an e-
commerce platform. 
• Implemented ARIMA and machine learning-based models to capture seasonal 
patterns and improve prediction accuracy. 
Technologies: Python, Pandas, Statsmodels, Scikit-learn 
Project 3: Retrieval-Augmented Generation (RAG) Chatbot 
• Designed a document-based question answering system leveraging

Chunk 3: • Designed a document-based q

### Embeddings

In [12]:
# Initialize OpenAI Embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key
)

# Test embedding
test_embedding = embeddings.embed_query("What is RAG?")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

Embedding dimension: 1536
First 5 values: [0.0006628383416682482, 0.025741448625922203, 0.007136902771890163, 0.03336041420698166, -0.03193381801247597]


### Vector Store

In [13]:
# Create vector store from documents
persist_directory = "./chroma_db"
collection_name = "collection"

if os.path.exists(persist_directory):
    vectorstore = Chroma(
        collection_name="collection",
        persist_directory=persist_directory,
        embedding_function=embeddings
    )
    print("chroma_db loaded")
else:
    vectorstore = Chroma.from_documents(
        chunks,
        embeddings,
        collection_name="my_info_collection",
        persist_directory="./chroma_db"
    )
    print("chroma_db created")

chroma_db created


In [14]:
# Test retriver
query = "What projects has Olasunkanmi worked on?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke(query)
results

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(doc.page_content)


Result 1:
I, Olasunkanmi Akeem Rasak, am an AI Engineer with a strong foundation in mathematics, data 
science, and applied machine learning, specializing in building intelligent systems that combine 
predictive modeling, natural language processing, and data-driven decision-making. My expertise spans 
real-world AI applications in finance, e-commerce, and education, with hands-on experience designing, 
training, and deploying scalable machine learning models.

Result 2:
Olasunkanmi Akeem Rasak 
AI Engineer | Data Scientist | Financial Engineer 
Professional Summary: 
AI Engineer and Data Scientist with hands-on experience designing, training, and 
deploying machine learning and deep learning models. Proficient in developing 
scalable AI solutions, predictive analytics systems, and NLP-based applications. 
Skilled in Python, SQL, PyTorch, LangChain, and cloud-based ML pipelines, with a 
strong foundation in applied mathematics and quantitative finance. Adept at

Result 3:
Current Loca

### Conversational RAG

In [15]:
# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=api_key
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# Store for chat histories
chat_store = {}

def get_session_history(session_id: str):
    if session_id not in chat_store:
        chat_store[session_id] = InMemoryChatMessageHistory()
    return chat_store[session_id]

# Create conversational prompt
conv_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the personal AI assistant for Olasunkanmi Akeem Rasak. "
     "You must answer questions ONLY if:\n"
     "1. The question is clear, complete, and meaningful.\n"
     "2. The provided context directly contains the information needed.\n\n"
     "If the question is unclear, incomplete, or nonsensical, respond exactly with:\n"
     "'Please ask a clear and complete question.'\n\n"
     "If the context does not contain the answer, respond exactly with:\n"
     "'I don’t have information about this in the provided documents.'\n\n"
     "Do NOT guess, infer, or generalize beyond the context."
    ),

    MessagesPlaceholder(variable_name="chat_history"),

    ("human",
     "Context:\n{context}\n\nQuestion:\n{question}"),

    ("system",
     "Rules for formatting the final answer:\n"
     "- Answer in clear, concise sentences.\n"
     "- List sources as bullet points ONLY if you actually used the context.\n"
     "- If you did not use the context, do NOT list any sources."
    )
])

# format documents
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in docs
    )

# Build base chain
conv_chain_base = (
    RunnableParallel(
        context=lambda x: format_docs(retriever.invoke(x["question"])),
        question=lambda x: x["question"],
        chat_history=lambda x: x.get("chat_history", [])
    )
    | conv_prompt
    | llm
    | StrOutputParser()
)

# Wrap with message history
conv_chain = RunnableWithMessageHistory(
    conv_chain_base,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)


#### Conversational RAG

### Questions!!!

In [23]:
# First question
response = conv_chain.invoke(
    {"question": "What projects has Olasunkanmi worked on?"},
    config={"configurable": {"session_id": "user_1"}}
)
print("Response 1:\n", response)

# Follow-up question
response2 = conv_chain.invoke(
    {"question": "Which of those projects are AI and Machine Learning projects?"},
    config={"configurable": {"session_id": "user_1"}}
)

print("\nResponse 2:\n", response2)

Response 1:
 Olasunkanmi Akeem Rasak has worked on AI projects involving sales forecasting, car price prediction, student performance analytics, forex trading strategy automation, and a hybrid car scanner.

Response 2:
 The AI and Machine Learning projects that Olasunkanmi Akeem Rasak has worked on include sales forecasting, car price prediction, student performance analytics, forex trading strategy automation, and a hybrid car scanner.


### Interactive Session

In [ ]:
session_id = "user_1"


while True:
    user_input = input("\nAsk your question about Olasunkanmi (or type 'exit' to quit): ")

    if user_input.lower() in ["exit", "quit", "bye"]:
        print(Markdown("### Goodbye! Have a blissful day."))
        break

    response = conv_chain.invoke(
        {"question": user_input},
        config={"configurable": {"session_id": session_id}}
    )

    # print("\nResponce:", response)
    display(Markdown(f"""
---
**User:**  
{user_input}

**Response:**  
{response}
"""))

clear_output(wait=True)
